In [ ]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv


In [ ]:
def print_json_structure(obj, indent=0):
    pad = "  " * indent

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{pad}{key}: {type(value).__name__}")
            print_json_structure(value, indent + 1)

    elif isinstance(obj, list):
        print(f"{pad}[list] len={len(obj)}")
        if obj:
            print_json_structure(obj[0], indent + 1)

In [ ]:
#data chmi

#https://opendata.chmi.cz/meteorology/climate/historical/data/1hour/



In [ ]:
def get_chmi_stations_metadata():
    url = 'https://opendata.chmi.cz/'
    route = '/meteorology/climate/historical/metadata/meta1.json'

    headers = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)
    response.raise_for_status()
    
    data = response.json()
    data = data['data']['data']

    headers = data['header'].split(',')
    values = data['values']

    df = pd.DataFrame(values, columns=headers)
    df.to_csv("data/chmi_stations_metadata.csv", index=False, encoding="utf-8-sig")

get_chmi_stations_metadata()


In [54]:
### data processing

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [ ]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open("data/wsi_dict.csv","w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [ ]:
### chmi data variables


In [ ]:
def get_chmi_variables_metadata():
    url = 'https://opendata.chmi.cz/'
    route = '/meteorology/climate/historical/metadata/meta2.json'

    headers = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)
    response.raise_for_status()

    response = response.json()

    data = response['data']['data']

    headers = data['header'].split(',')
    values = data['values']

    df = pd.DataFrame(values, columns=headers)
    df.to_csv("data/chmi_variables_metadata.csv", index=False, encoding="utf-8-sig")

get_chmi_variables_metadata()

In [ ]:
### filtering only needed ones

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

In [ ]:
### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

In [ ]:
### golemio data 


##### air quality stations 

## metadata: https://opendata.chmi.cz//air_quality/recent/metadata/metadata.json 

In [ ]:
def set_api_key(token = 'api_key.env'):
    load_dotenv(token)
    api_key = os.getenv('GOLEMIO_API_KEY')
    if api_key:
        print("API key loaded")
    else:
        print("API key not found")
    return api_key

api_key = set_api_key()


✓ API key loaded


In [ ]:
def get_airquality_stations_metadata():
    url = 'https://api.golemio.cz/'
    route = '/v2/airqualitystations'

    headers = {
        'X-access-token': api_key, 
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)
    response.raise_for_status()

    data = response.json()
    df = pd.json_normalize(data['features'])

    tmp = df.explode("properties.measurement.components", ignore_index=True)
    dirs = pd.json_normalize(
        tmp["properties.measurement.components"]
    ).add_prefix("properties.measurement.components.")

    df = pd.concat([tmp.drop(columns=["properties.measurement.components"]), dirs], axis=1)

    df.to_csv("data/airquality_stations_metadata.csv", index=False, encoding="utf-8-sig")

get_airquality_stations_metadata()

In [ ]:
#### air quality metadata processing

station_cols = {
    'geometry.coordinates': 'coordinates', 
    'properties.id': 'id', 
    'properties.name': 'name', 
    'properties.district': 'district', 
    'properties.measurement.components.type': 'components'
}

In [209]:
air_quality_stations = df[station_cols.keys()]

air_quality_stations = air_quality_stations.rename(columns=station_cols)

In [211]:
air_quality_stations = (
    air_quality_stations.groupby('id', as_index=False)
    .agg({
        'coordinates': 'first',
        'name': 'first',
        'district': 'first',
        'components': list
    })
)

In [216]:
air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")

In [ ]:
#air quality stations dictionary

air_stat_dict = dict(
    zip(
        air_quality_stations['id'].astype(str),
        air_quality_stations['name'].astype(str)  
    )
)

In [ ]:
### chmi weather data

# 10 min data download

In [ ]:
def get_chmi_weather_data(start_year = 2025, end_year = 2025):
    base_url = "https://opendata.chmi.cz/"
    route_template = "/meteorology/climate/historical/data/10min/{year}/10m-{wsi}-{ym}.json"

    url_header = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }
    
    wsi_dict = pd.read_csv("wsi_dict.csv", encoding="utf-8-sig").set_index("key")["value"].to_dict()

    years = [f"{y}" for y in range(start_year, end_year+1)]
    months = [f"{m:02d}" for m in range(1, 13)]

    results = []

    for wsi in wsi_dict:
        for year in years:
            for month in months:
                ym = f"{year}{month}"
                route = route_template.format(year=year, wsi=wsi, ym=ym)

            try:
                response = requests.get(f"{base_url}{route}", headers=url_header, timeout=60)
                response.raise_for_status()
            except requests.exceptions.RequestException as exc:
                print(f"Request failed for {wsi} {wsi_dict.get(wsi)} {year} {month}: {exc}")
                continue

            response = response.json()
            data_response = response.get('data', {}).get('data', {})

            headers = data_response.get('header', '').split(',')
            values = data_response.get('values', [])

            df_part = pd.DataFrame(values, columns=headers)
            df_part["WSI"] = wsi
            df_part["YEAR"] = year
            df_part["MONTH"] = month

            results.append(df_part)

    df = pd.concat(results, ignore_index=True)
    df.to_csv("data/weather_data_10min.csv", index=False, encoding="utf-8-sig")


get_chmi_weather_data()

Failed for 0-203-0-11201020001 Praha, Vinohrady - Flora 2025 03: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 01: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 02: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 03: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 04: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 05: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 06: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 07: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 08: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 09: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 10: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 11: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 12: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 01: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 02: 404
Failed for 0-203-0-11

In [ ]:
### weather data for 1hour -- if needed

base_url = "https://opendata.chmi.cz/"
route_template = "/meteorology/climate/historical/data/1hour/{year}/1h-{wsi}-{ym}.json"

url_header = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

years = [2025]
months = [11] #[f"{m:02d}" for m in range(1, 13)]

results = []

for wsi in wsi_dict:
    for year in years:
        for month in months:
            ym = f"{year}{month}"
            route = route_template.format(year=year, wsi=wsi, ym=ym)

            try:
                response = requests.get(f"{base_url}{route}", headers=url_header, timeout=60)
                response.raise_for_status()
            except requests.exceptions.RequestException as exc:
                print(f"Request failed for {wsi} {wsi_dict.get(wsi)} {year} {month}: {exc}")
                continue

            data_response = response.json()
            data_response = data_response['data']['data']

            headers = data_response['header'].split(',')
            values = data_response['values']

            df_part = pd.DataFrame(values, columns=headers)
            df_part["WSI"] = wsi
            df_part["YEAR"] = year
            df_part["MONTH"] = month

            results.append(df_part)




# df_chmi = pd.concat(results, ignore_index=True)


Failed for 0-203-0-11515 Praha, Klementinum 2025 11: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 11: 404
Failed for 0-203-0-11101007001 Praha, Brdy 2025 11: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 11: 404
Failed for 0-203-0-11201020003 Praha, Chodov 2025 11: 404


In [ ]:
### mapping variables names

df_chmi['ELEMENT_NAME'] = df_chmi['ELEMENT'].map(chmi_vars_dict)

NameError: name 'chmi_vars_dict' is not defined

In [ ]:
### golemio 

### air quality data 

In [ ]:
api_url = 'https://api.golemio.cz/'
route = '/v2/airqualitystations/history'

headers = {
    'X-access-token': api_key, 
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

params = {
    'from': '2024-05-16T04:27:58.000Z', 
    'to': '2024-06-18T04:27:58.000Z'
}

response = requests.get(f'{api_url}{route}', headers=headers, params=params, timeout=60)

data = response.json()



In [243]:
print(json.dumps(data, indent=2))

{
  "error_message": "Not Found",
  "error_status": 404
}


In [ ]:
if data:
    print("Top level keys:", data[0].keys())
    print("Measurement level keys:", data[0]['measurement'].keys())

Top level keys: dict_keys(['id', 'measurement'])
Measurement level keys: dict_keys(['AQ_hourly_index', 'components'])


In [228]:
df = pd.json_normalize(
    data,
    record_path=["measurement", "components"],
    meta=[
        "id",
        ["measurement", "AQ_hourly_index"],
    ],
    sep=".",
    errors="raise",
)

In [ ]:
#### aternative air quality - chmi air quality

def download_latest_data(data_dir_url):
    response = requests.get(data_dir_url, timeout=60)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    csv_files = [link.get('href') for link in soup.find_all('a') if link.get('href', '').endswith('.csv')]
    
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the directory.")
    
    # sorting them alphabetically - the last one is the newest
    latest_file = sorted(csv_files)[-1]
    file_url = f"{data_dir_url}{latest_file}"
    
    print(f"Downloading raw data from: {latest_file}")
    data_response = requests.get(file_url)
    data_response.raise_for_status()

    df_data = pd.read_csv(io.StringIO(data_response.text))
    df_data.to_csv("data/airquality_CHMI_stations_data.csv", index=False, encoding="utf-8-sig")



In [ ]:
def download_metadata(metadata_url):
    response = requests.get(metadata_url, timeout=60)
    response.raise_for_status()
    metadata = response.json()

    mapping_list = []

    localities = metadata.get("data", {}).get("Localities", [])

    for locality in localities:
        lon = locality.get("Localization").get("LonAsNumber")
        lat = locality.get("Localization").get("LatAsNumber")
        alt = locality.get("Localization").get("Alt")
        street = locality.get("Address").get("Street")
        city = locality.get("Address").get("City")
        programs = locality.get("MeasuringPrograms")
        for program in programs:
            station_code = program.get("Code")
            measurements = program.get("Measurements")
            for measurement in measurements:
                row = {
                    "id_registration": measurement.get("IdRegistration"),
                    "station_code": station_code,
                    "street": street,
                    "city": city, 
                    "lon": lon, 
                    "lat": lat,
                    "alt": alt,
                    "component_code": measurement.get("ComponentCode"),
                    "component_name": measurement.get("ComponentName"),
                    "unit": measurement.get("UnitAsASCII")
                }
                mapping_list.append(row)

    df_mapping = pd.DataFrame(mapping_list)
    df_mapping.to_csv("data/airquality_CHMI_stations_metadata.csv", index=False, encoding="utf-8-sig")
    

In [ ]:

META_URL = "https://opendata.chmi.cz/air_quality/recent/metadata/metadata.json"
DATA_URL = "https://opendata.chmi.cz/air_quality/recent/data/"

download_metadata(META_URL)
download_latest_data(DATA_URL)


In [18]:
#### golemio

### microclimate sensors

In [19]:
# TODO if necessary